# Reproduce `refusal_direction` on Gemma-2B (Colab, free T4 GPU)

Runs the full pipeline from *Refusal in Language Models Is Mediated by a Single Direction* on a free Colab GPU in ~20–45 min, giving the paper's headline metrics for `google/gemma-2b-it`.

It clones the upstream repo and applies **one small patch** (makes the CUDA-only `vllm` import lazy) so you avoid installing `vllm` on Colab. My Apple-Silicon fixes are Mac-only and not needed here.

## Before you run

1. **Runtime → Change runtime type → T4 GPU.**
2. Logged in to HuggingFace (the account tied to your `HF_TOKEN`), accept these licenses:
   - https://huggingface.co/google/gemma-2b-it — *Acknowledge license*
   - https://huggingface.co/meta-llama/Meta-Llama-Guard-2-8B — *Acknowledge license* (used by the `llamaguard2` scorer; skip if you run the substring-only option below)
3. Add two **Colab Secrets** (🔑 icon, left sidebar): `HF_TOKEN` and `TOGETHER_API_KEY`.

### 1. Confirm the GPU is attached

In [ ]:
!nvidia-smi -L   # expect a Tesla T4 (or better)

### 2. Clone the repo + install a light dependency set

Colab already ships CUDA-enabled `torch`, so we skip the heavy CUDA-only packages (`vllm`, `xformers`, `flash-attn`, `nvidia-*`).

In [ ]:
!git clone https://github.com/andyrdt/refusal_direction.git
%cd refusal_direction
!pip -q install "transformers==4.44.2" "accelerate>=0.31" "datasets==2.19.2" litellm jaxtyping einops sentencepiece matplotlib

### 3. Patch `vllm` to be imported lazily

`vllm` is only used by the optional `harmbench` evaluator, but upstream imports it at module top-level, which crashes when `vllm` isn't installed. This makes the import lazy (same fix used for the Mac port).

In [ ]:
import pathlib
p = pathlib.Path("pipeline/submodules/evaluate_jailbreak.py")
s = p.read_text()
s = s.replace("from vllm import LLM, SamplingParams\n", "")
s = s.replace("from vllm.distributed.parallel_state import destroy_model_parallel\n", "")
s = s.replace(
    "def harmbench_judge_fn(prompts: List[str], responses: List[str]) -> List[int]:\n",
    "def harmbench_judge_fn(prompts: List[str], responses: List[str]) -> List[int]:\n"
    "    from vllm import LLM, SamplingParams\n"
    "    from vllm.distributed.parallel_state import destroy_model_parallel\n",
)
p.write_text(s)
print("patched: vllm import is now lazy")

### 4. Authenticate (HuggingFace + Together)

Reads the two Colab Secrets you added. `TOGETHER_API_KEY` is used for the `llamaguard2` safety scorer.

In [ ]:
import os
from google.colab import userdata
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
os.environ["TOGETHER_API_KEY"] = userdata.get("TOGETHER_API_KEY")
from huggingface_hub import login
login(os.environ["HF_TOKEN"])
print("authenticated")

### 5. (Optional) Substring-matching eval only

Run this cell to skip the gated LlamaGuard-2 scorer for a first pass — no second license and no Together API needed. Skip it to keep the default (`substring_matching` + `llamaguard2`).

In [ ]:
import pathlib
c = pathlib.Path("pipeline/config.py")
t = c.read_text()
t = t.replace(
    'jailbreak_eval_methodologies: Tuple[str] = ("substring_matching", "llamaguard2")',
    'jailbreak_eval_methodologies: Tuple[str] = ("substring_matching",)',
)
c.write_text(t)
print("eval set to substring_matching only")

### 6. Run the full pipeline (~20–45 min)

Extracts candidate directions → selects the best refusal direction → generates baseline / ablation / act-add completions → evaluates refusal + CE-loss metrics.

In [ ]:
!python3 -m pipeline.run_pipeline --model_path google/gemma-2b-it

### 7. Save results (Colab storage is ephemeral)

Artifacts are in `pipeline/runs/gemma-2b-it/` — `direction.pt`, selection plots, completions, and `*_evaluations.json` (the ASR / refusal metrics). Zip + download them, or mount Google Drive to persist across sessions.

In [ ]:
!zip -rq gemma_results.zip pipeline/runs/gemma-2b-it
from google.colab import files
files.download("gemma_results.zip")